<a href="https://colab.research.google.com/github/wtraquinas/lab-agent-vector-store/blob/main/solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab | Agent & Vector store

# Solution :

<br>

---

## 🚀 Your turn

Time to make this lab your own!

Replace the `state_of_the_union.txt` dataset with something you'd actually enjoy chatting with. A great place to start is the [sonnets.txt dataset](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) —or any other .txt file from that repository. Of course, you're not limited to those options. Pick any text that interests you and see how your chatbot responds.

Here's what to do:

1. **Get your data** — Download your chosen `.txt` file into this project folder (or point `TextLoader` at it directly).
2. **Rebuild the vector store** — Load, split, and embed your new document, then create a fresh `RetrievalQA` chain for it (give it a descriptive `collection_name`!).
3. **Rewrite the tool description** — Update the `Tool`'s `name` and `description` so the agent knows *when* it should reach for this new tool instead of the Ruff or state-of-the-union ones.
4. **Rebuild the agent** — Combine your new tool with the existing Ruff tool (or drop it if you'd rather keep just your new dataset + one other source).
5. **Put it to the test** — Ask your agent:
   - A direct question that only your new dataset can answer.
   - A question that only the Ruff tool can answer.
   - A multi-hop question that requires combining *both* tools' knowledge, like the Jupyter/Ruff example above.
6. **Reflect** — In a markdown cell, briefly note whether the agent picked the right tool(s) each time, and what happened when you set `return_direct=True` vs. not.

⭐️ **Bonus points:**
- Instead of modifying this same file, create a new file `solution.ipynb` and replicate the process from scratch.

💡 **Tip:**
- Watch the `verbose=True` agent logs closely — they show you the agent's reasoning step by step, which is the best way to understand *why* it picked a particular tool.





---



In [1]:
# -- Uncomment and run the cells below
# -- to install the required dependencies for this notebook.

!pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"
!pip install python-dotenv==1.2.2 chromadb==1.5.9 beautifulsoup4==4.15.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.1.4
    Uninstalling tenacity-9.1.4:
      Successfully uninstalled tenacity-9.1.4
  Attempting uninstall: packaging
    Found exis

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 140.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

In [1]:
# -- Before building anything, we need to load our API credentials
# -- and instantiate the LLM we'll use throughout the notebook

from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

from google.colab import userdata
OPENAI_API_KEY  = userdata.get('OPENAI_API_KEY')

llm = OpenAI(temperature=0, api_key=OPENAI_API_KEY)

## 1. Get your data
- Download your chosen .txt file into this project folder (or point TextLoader at it directly).

In [2]:
doc_path =  "./datasets/juliuscaesar.txt"

## 2. Rebuild the vector store
- Load, split, and embed your new document, then create a fresh RetrievalQA chain for it (give it a descriptive collection_name!).

In [3]:
loader = TextLoader(doc_path)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

docsearch = Chroma.from_documents(texts, embeddings, collection_name="julius_caesar_sonnet")

## 3. Rewrite the tool description
- Update the Tool's name and description so the agent knows when it should reach for this new tool instead of the Ruff or state-of-the-union ones.

In [4]:
julius_caesar_sonnet = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [5]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

In [6]:
# Import things that are needed generically
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import OpenAI

In [7]:
tools = [
    Tool(
        name="Julius Caesar Sonnet QA System",
        func=julius_caesar_sonnet.run,
        description="useful for when you need to answer questions about Shakespear's Julius Caesar Sonnet. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

## 4. Rebuild the agent
- Combine your new tool with the existing Ruff tool (or drop it if you'd rather keep just your new dataset + one other source).

In [8]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

/tmp/ipykernel_8677/1834837320.py:3: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


## 5. Put it to the test
- Ask your agent:
  - A direct question that only your new dataset can answer.
  - A question that only the Ruff tool can answer.
  - A multi-hop question that requires combining both tools' knowledge, like the Jupyter/Ruff example above.

In [9]:
agent.invoke(
    "What did Brutus say about the Ides of March?"
)

 Brutus is a character in Julius Caesar, so I should use the Julius Caesar Sonnet QA System.
Action: Julius Caesar Sonnet QA System
Action Input: "What did Brutus say about the Ides of March?"
Observation:  Brutus said that the Ides of March was the day that they began their plan to kill Caesar and that it would also be the day that their work would be completed.
Thought: I now know the final answer.
Final Answer: Brutus said that the Ides of March was the day that they began their plan to kill Caesar and that it would also be the day that their work would be completed.

> Finished chain.


{'input': 'What did Brutus say about the Ides of March?',
 'output': 'Brutus said that the Ides of March was the day that they began their plan to kill Caesar and that it would also be the day that their work would be completed.'}

In [10]:
agent.invoke("Why use ruff over flake8?")

 You should consider the differences between ruff and flake8 before making a decision.
Action: Ruff QA System
Action Input: "What are the differences between ruff and flake8?"
Observation:  Ruff has a larger rule set and does not support custom lint rules, while Flake8 supports plugins and allows for custom and third-party rules. Ruff also has a formatter and can automatically fix its own lint violations, while Flake8 does not have these capabilities. Additionally, Ruff is written in Rust while Flake8 is written in Python.
Thought: I now have a better understanding of the differences between ruff and flake8.
Action: Ruff QA System
Action Input: "Why should I use ruff instead of flake8?"
Observation:  Ruff offers a larger rule set, automatic fixing of lint violations, and compatibility with Black. It also re-implements popular Flake8 plugins and related code quality tools natively. Additionally, Ruff supports Python versions from 3.7 onwards, while Flake8 only supports Python 2. Ruff al

{'input': 'Why use ruff over flake8?',
 'output': 'Based on the information provided by the Ruff QA System, it seems that ruff offers more features and compatibility with newer versions of Python, making it a better choice for linting. However, the decision ultimately depends on the specific needs and preferences of the user.'}

In [11]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did Caesar mention that tool in his speechs?"
)

 I should use the Ruff QA System to answer this question.
Action: Ruff QA System
Action Input: What tool does ruff use to run over Jupyter Notebooks?
Observation:  Ruff uses nbQA, a tool for running linters and code formatters over Jupyter Notebooks.
Thought: I should use the Julius Caesar Sonnet QA System to check if Caesar mentioned this tool in his speeches.
Action: Julius Caesar Sonnet QA System
Action Input: Did Caesar mention nbQA in his speeches?
Observation:  No, Caesar did not mention nbQA in his speeches.
Thought: I now know the final answer.
Final Answer: No, Caesar did not mention nbQA in his speeches.

> Finished chain.


{'input': 'What tool does ruff use to run over Jupyter Notebooks? Did Caesar mention that tool in his speechs?',
 'output': 'No, Caesar did not mention nbQA in his speeches.'}

## 6.Reflect  
- In a markdown cell, briefly note whether the agent picked the right tool(s) each time, and what happened when you set return_direct=True vs. not.

In [12]:
tools = [
    Tool(
        name="Julius Caesar Sonnet QA System",
        func=julius_caesar_sonnet.run,
        description="useful for when you need to answer questions about Shakespear's Julius Caesar Sonnet. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [13]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [14]:
agent.invoke(
    "What did Brutus say about the Ides of March?"
)

 Brutus is a character in Julius Caesar, so I should use the Julius Caesar Sonnet QA System.
Action: Julius Caesar Sonnet QA System
Action Input: "What did Brutus say about the Ides of March?"
Observation:  Brutus said that the Ides of March was the day that they began their plan to kill Caesar and that it would also be the day that their work would be completed.


> Finished chain.


{'input': 'What did Brutus say about the Ides of March?',
 'output': ' Brutus said that the Ides of March was the day that they began their plan to kill Caesar and that it would also be the day that their work would be completed.'}

In [15]:
agent.invoke("Why use ruff over flake8?")

 You should consider the differences between ruff and flake8 before making a decision.
Action: Ruff QA System
Action Input: "What are the differences between ruff and flake8?"
Observation:  Ruff has a larger rule set and does not support custom lint rules, while Flake8 supports plugins and allows for custom and third-party rules. Ruff also has a formatter and can automatically fix its own lint violations, while Flake8 does not have these capabilities. Additionally, Ruff is written in Rust while Flake8 is written in Python.


> Finished chain.


{'input': 'Why use ruff over flake8?',
 'output': ' Ruff has a larger rule set and does not support custom lint rules, while Flake8 supports plugins and allows for custom and third-party rules. Ruff also has a formatter and can automatically fix its own lint violations, while Flake8 does not have these capabilities. Additionally, Ruff is written in Rust while Flake8 is written in Python.'}

In [16]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did Caesar mention that tool in his speechs?"
)

 I should use the Ruff QA System to answer this question.
Action: Ruff QA System
Action Input: What tool does ruff use to run over Jupyter Notebooks?
Observation:  Ruff uses nbQA, a tool for running linters and code formatters over Jupyter Notebooks.


> Finished chain.


{'input': 'What tool does ruff use to run over Jupyter Notebooks? Did Caesar mention that tool in his speechs?',
 'output': ' Ruff uses nbQA, a tool for running linters and code formatters over Jupyter Notebooks.'}

The agent answered the first 2 questions the same way with return_direct=True or False, because the queries were single-step in nature, but for the third multi-hop question, the result was very different when using return_direct=True sunce it needed the reasoning loop to acess both datasets for the correct answer.

- By default (return_direct=False), when a tool returns an Observation, the agent takes that Observation back into its reasoning loop and generates its own Final Answer wrapping/summarizing it.

- With return_direct=True on a Tool, the agent skips that final synthesis step — whatever the tool function returns is passed straight back to the user verbatim, and the Thought/Action loop ends immediately after that one tool call.

- Practical takeaway: `return_direct=True` trades flexibility (no multi-tool
  chaining, no natural-language polish) for speed and determinism — worth
  using on a tool only when you're confident every query needing it is
  single-step and the raw output is already user-ready.